# 05, VLM LED extraction

**NOT EXECUTED IN THIS ENVIRONMENT**, Track B/C below need a running Ollama server with
`llava:7b` (and optionally `llava:13b`) pulled. The heuristic extractor (Track A) has no such
dependency and *could* run here, but is kept alongside the others, unexecuted, so the three-way
comparison in this notebook stays apples-to-apples in one place.

**Dataset correction (verified by inspection, not assumed):** `data/router
detection.v38-data_video.coco.zip` is a **cable/port/connector** detection dataset, categories
are `{fiber,lan,phone,power,usb} x {cable,conn}` (2137 images / 4630 boxes), not LED-colour
states. See `evaluation/datasets/roboflow_router.py`. It is used here only as a *structured hint*
source (which ports/cables are visibly connected) injected into the VLM prompt, the LED
colour/behaviour ground truth below is a small, separately hand-labelled set
(`data/processed/led_labels.jsonl`), built from `ingestion-pipeline-prototype/data/sample_images/`
plus this notebook's target of 30-60 images once more router photos are available.

**Inputs:** `data/router detection.v38-data_video.coco.zip`,
`ingestion-pipeline-prototype/data/sample_images/*.png`, `models/prompts/vlm/router_led_panel.txt`

**Outputs:** `data/processed/led_labels.jsonl`, the field-accuracy comparison table below, the
chosen `vlm_impl` + tuned prompt + confidence threshold (feeds `config/settings.yaml`).

**Pitfall:** Ollama vision needs the image as base64 **without** a `data:` prefix; a 400 "invalid
image" almost always means the prefix was left in, or the file is a PNG with an alpha channel
that PIL should have re-saved as RGB first (`libs/platform/models/vlm.py::_ensure_rgb_png_bytes`
already does this for the real adapter, this notebook's ad-hoc encoding below must too).

In [ ]:
import os, sys
from pathlib import Path

# Notebook hygiene: always run from the v1 root, never hard-code absolute d:\... paths.
_here = Path.cwd()
_v1_root = _here if (_here / "pyproject.toml").exists() else _here.parents[0]
os.chdir(_v1_root)
sys.path.insert(0, str(_v1_root))

# Notebooks that touch libs.platform.registry/config must pin a profile explicitly, the
# process env has no APP_PROFILE by default, which silently falls back to "cpu" (real Ollama
# calls, no stub) and produces confusing failures/latency. Pin stub here; override the env var
# before importing libs.platform.config in the rare notebook that needs otherwise.
os.environ.setdefault("APP_PROFILE", "stub")
print("cwd:", Path.cwd(), "| APP_PROFILE:", os.environ["APP_PROFILE"])

In [ ]:
import base64
import json

from evaluation.datasets.roboflow_router import load_split

train_split = load_split("train")
print(f"{len(train_split.images)} images, {len(train_split.annotations)} boxes, categories: {list(train_split.categories.values())}")

## Hand-labelled LED ground truth

30-60 images hand-labelled into `data/processed/led_labels.jsonl`. This environment only has 2
router photos available (`router_red_led.png`, `router_green_led.png` in the prototype's sample
data), both labelled below as a seed; the remaining ~28-58 need real router photos, out of
scope to source here.

In [ ]:
PROTOTYPE_IMAGES = Path("../ingestion-pipeline-prototype/data/sample_images")

LED_LABELS_SEED = [
    {
        "file": "router_red_led.png",
        "device_model": "unknown",
        "led_states": [{"label": "power", "colour": "red", "behaviour": "solid"}],
    },
    {
        "file": "router_green_led.png",
        "device_model": "unknown",
        "led_states": [{"label": "power", "colour": "green", "behaviour": "solid"}],
    },
]

out_path = Path("data/processed/led_labels.jsonl")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    for row in LED_LABELS_SEED:
        f.write(json.dumps(row) + "\n")
print(f"wrote {len(LED_LABELS_SEED)} seed labels to {out_path} (target: 30-60 for a full run)")

## Track A: heuristic HSV extractor (no Ollama needed, could run here, kept unexecuted for a fair three-way comparison)

In [ ]:
from libs.platform.models.vlm import HeuristicLedExtractor

heuristic = HeuristicLedExtractor()
track_a_rows = []
for label in LED_LABELS_SEED:
    path = PROTOTYPE_IMAGES / label["file"]
    if not path.exists():
        continue
    summary = heuristic.extract(path, attachment_id=label["file"], template="router_led_panel")
    predicted_colours = {led.colour for led in summary.extracted_fields.led_states}
    expected_colours = {led["colour"] for led in label["led_states"]}
    track_a_rows.append({"file": label["file"], "predicted": predicted_colours, "expected": expected_colours, "match": bool(predicted_colours & expected_colours)})

track_a_rows

## Track B: `llava:7b` via Ollama

In [ ]:
from libs.platform.models.vlm import OllamaVisionExtractor

llava_7b = OllamaVisionExtractor(base_url="http://localhost:11434", model="llava:7b")
track_b_rows = []
for label in LED_LABELS_SEED:
    path = PROTOTYPE_IMAGES / label["file"]
    if not path.exists():
        continue
    summary = llava_7b.extract(path, attachment_id=label["file"], template="router_led_panel")
    predicted_colours = {led.colour for led in summary.extracted_fields.led_states}
    expected_colours = {led["colour"] for led in label["led_states"]}
    track_b_rows.append({
        "file": label["file"], "predicted": predicted_colours, "expected": expected_colours,
        "match": bool(predicted_colours & expected_colours), "confidence": summary.confidence,
    })

track_b_rows

## Track C: `llava:13b` via Ollama, if it fits

In [ ]:
llava_13b = OllamaVisionExtractor(base_url="http://localhost:11434", model="llava:13b")
track_c_rows = []
for label in LED_LABELS_SEED:
    path = PROTOTYPE_IMAGES / label["file"]
    if not path.exists():
        continue
    try:
        summary = llava_13b.extract(path, attachment_id=label["file"], template="router_led_panel")
        predicted_colours = {led.colour for led in summary.extracted_fields.led_states}
        expected_colours = {led["colour"] for led in label["led_states"]}
        track_c_rows.append({
            "file": label["file"], "predicted": predicted_colours, "expected": expected_colours,
            "match": bool(predicted_colours & expected_colours), "confidence": summary.confidence,
        })
    except Exception as exc:
        print(f"llava:13b unavailable or failed ({exc}); skipping Track C")
        break

## Structured hint: injecting port/cable detections into the VLM prompt

The Roboflow dataset's real value in this pipeline is as a *pre-filter*: run a small detector
over the image first, and inject "the following ports/cables are visibly connected: ..." into
the VLM prompt, so the VLM's job narrows from "describe the whole board" to "confirm/refine what
the detector already found". A full YOLO/Detectron training run is out of scope for this
notebook; sketched below is the prompt-injection shape a trained detector's output would take.

In [ ]:
def inject_structured_hint(base_prompt: str, detected_labels: list[str]) -> str:
    if not detected_labels:
        return base_prompt
    hint = "Detected on this board (from an object detector, confirm or correct): " + ", ".join(detected_labels)
    return f"{hint}\n\n{base_prompt}"


base_prompt = Path("models/prompts/vlm/router_led_panel.txt").read_text(encoding="utf-8")
example_hint = inject_structured_hint(base_prompt, ["power-conn", "lan-cable"])
print(example_hint[:300])

## Chosen configuration (to be finalised once Track B/C actually run)

Placeholder pending a real run: default to `vlm_impl=heuristic` (Track A) as the always-available
fallback per the SAD deviation table, and switch to `vlm_impl=ollama` with `llava:7b` once Track
B's field accuracy is measured to beat it on a real 30-60 image labelled set.